# nb_01_ingest_files
**SYNTHETIC DATA · for training only · no real customer data**

Bronze ingestion for the FGD security demo:
1. **Simulated external drop** - writes 3 daily `loan_applications_YYYY-MM-DD.csv` files (~1,000 rows each) to `lh_bronze` `Files/landing/loan_applications/`.
2. **Load** - appends new files to the Delta table `loan_applications_raw` (adds `ingest_ts`, `source_file`). Idempotent by file name.
3. **OneLake shortcuts** - creates `customers` and `accounts` shortcuts in `lh_bronze/Tables` pointing at the tables that SQL database `sqldb_core_banking` automatically replicates to OneLake (`Tables/dbo/<table>`). No data is copied.

Run by: pipeline `pl_daily_load` (first activity) or manually by a Contributor (dana / presenter).

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
# nb_01_ingest_files - parameters (tagged "parameters" so the pipeline can override them)
workspace_name        = ""                      # "" = the workspace this notebook runs in
bronze_name           = "lh_bronze"
sqldb_name            = "sqldb_core_banking"
landing_folder        = "Files/landing/loan_applications"
raw_table             = "loan_applications_raw"
file_dates            = ["2026-10-28", "2026-10-29", "2026-10-30"]   # 3 daily files
rows_per_file         = 1000                    # ~3,000 rows total
max_customer_id       = 5000                    # matches sql/10_sqldb_seed.sql
shortcut_tables       = ["customers", "accounts"]
shortcut_wait_minutes = 15                      # wait for SQL DB -> OneLake replication

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data

# Resolve workspace and item ids with semantic-link (sempy) so this notebook does
# NOT depend on a default lakehouse. All paths below are absolute abfss:// URLs.
import sempy.fabric as fabric
import notebookutils

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def resolve_ws(name):
    """Workspace id for a name; '' means the workspace this notebook runs in."""
    if not name:
        return fabric.get_workspace_id()
    try:
        return fabric.resolve_workspace_id(name)
    except AttributeError:  # older semantic-link versions
        df = fabric.list_workspaces()
        return df[df["Name"] == name]["Id"].iloc[0]

def resolve_item(name, item_type, ws_id):
    """Item id for a display name + type (Lakehouse, SQLDatabase, ...)."""
    try:
        return fabric.resolve_item_id(name, type=item_type, workspace=ws_id)
    except Exception:
        df = fabric.list_items(type=item_type, workspace=ws_id)
        hit = df[df["Display Name"] == name]
        if hit.empty:
            raise ValueError(f"{item_type} '{name}' not found in workspace {ws_id}")
        return hit["Id"].iloc[0]

def item_root(ws_id, item_id):
    return f"abfss://{ws_id}@{ONELAKE}/{item_id}"

ws_id     = resolve_ws(workspace_name)
bronze_id = resolve_item(bronze_name, "Lakehouse", ws_id)
sqldb_id  = resolve_item(sqldb_name, "SQLDatabase", ws_id)

bronze_root = item_root(ws_id, bronze_id)
sqldb_root  = item_root(ws_id, sqldb_id)
landing     = f"{bronze_root}/{landing_folder}"
raw_path    = f"{bronze_root}/Tables/{raw_table}"

print(f"[step] workspace  : {ws_id}")
print(f"[step] lh_bronze  : {bronze_id}")
print(f"[step] sqldb      : {sqldb_id}")
print(f"[step] landing    : {landing}")

## 1 · Simulated external file drop
In real life an upstream system (or a Data Factory copy job) drops files here. We generate them deterministically so every tenant gets identical data. Existing files are never overwritten.

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
import csv, io, random
from datetime import datetime, timedelta

# product, min amount, max amount, min rate, max rate
PRODUCTS = [
    ("Mortgage",       150000, 900000, 4.20,  6.50),
    ("HELOC",           20000, 250000, 6.00,  8.50),
    ("Personal Loan",    2000,  50000, 7.50, 13.90),
    ("Auto Loan",        8000,  75000, 5.50,  9.90),
    ("Line of Credit",   5000, 100000, 7.00, 11.50),
]
STATUSES = ["Submitted", "Approved", "Declined", "Withdrawn"]
STATUS_WEIGHTS = [0.30, 0.40, 0.20, 0.10]

notebookutils.fs.mkdirs(landing)

for d in file_dates:
    fname = f"loan_applications_{d}.csv"
    path = f"{landing}/{fname}"
    if notebookutils.fs.exists(path):
        print(f"[step] {fname} already landed - skipped")
        continue

    rnd = random.Random(f"loan-{d}")          # deterministic per file
    day = datetime.strptime(d, "%Y-%m-%d")
    buf = io.StringIO()
    writer = csv.writer(buf, lineterminator="\n")
    writer.writerow(["application_id", "customer_id", "product", "amount", "rate", "status", "submitted_at"])
    for i in range(1, rows_per_file + 1):
        product, lo, hi, rlo, rhi = rnd.choice(PRODUCTS)
        writer.writerow([
            f"LA-{day:%Y%m%d}-{i:05d}",
            rnd.randint(1, max_customer_id),
            product,
            f"{rnd.uniform(lo, hi):.2f}",
            f"{rnd.uniform(rlo, rhi):.3f}",
            rnd.choices(STATUSES, weights=STATUS_WEIGHTS)[0],
            (day + timedelta(seconds=rnd.randint(0, 86399))).strftime("%Y-%m-%d %H:%M:%S"),
        ])
    notebookutils.fs.put(path, buf.getvalue(), True)
    print(f"[step] landed {fname} ({rows_per_file} rows)")

## 2 · Load new files into `loan_applications_raw`
Idempotent: files whose name is already in `source_file` are skipped.

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

schema = T.StructType([
    T.StructField("application_id", T.StringType()),
    T.StructField("customer_id",    T.IntegerType()),
    T.StructField("product",        T.StringType()),
    T.StructField("amount",         T.DecimalType(18, 2)),
    T.StructField("rate",           T.DecimalType(6, 3)),
    T.StructField("status",         T.StringType()),
    T.StructField("submitted_at",   T.TimestampType()),
])

already_loaded = set()
if DeltaTable.isDeltaTable(spark, raw_path):
    already_loaded = {r.source_file for r in
                      spark.read.format("delta").load(raw_path).select("source_file").distinct().collect()}

landed = sorted([f for f in notebookutils.fs.ls(landing) if f.name.endswith(".csv")], key=lambda f: f.name)
new_files = [f for f in landed if f.name not in already_loaded]
print(f"[step] {len(landed)} files landed, {len(new_files)} new")

for f in new_files:
    df = (spark.read
              .option("header", True)
              .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
              .schema(schema)
              .csv(f.path)
              .withColumn("ingest_ts", F.current_timestamp())
              .withColumn("source_file", F.lit(f.name)))
    df.write.format("delta").mode("append").save(raw_path)
    print(f"[step] loaded {f.name}: {df.count()} rows")

print(f"[step] {raw_table} total rows: {spark.read.format('delta').load(raw_path).count()}")

## 3 · OneLake shortcuts to the replicated SQL database tables
SQL database in Fabric replicates every table with a primary key to OneLake automatically (Delta, under `Tables/dbo/<table>`). A shortcut makes those tables appear in `lh_bronze` without copying data.

**UI fallback** (if this cell fails): open `lh_bronze` → `Tables` → `...` → **New shortcut** → **Microsoft OneLake** → pick `sqldb_core_banking` → expand `Tables/dbo` → tick `customers` and `accounts` → **Next** → **Create**.

Note: shortcuts inherit security from the *target*. A user who can't read `sqldb_core_banking` can't read through the shortcut.

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
import time

client = fabric.FabricRestClient()   # calls the Fabric REST API as the notebook user

def target_ready(table):
    """True once the SQL database has replicated the table to OneLake."""
    try:
        return notebookutils.fs.exists(f"{sqldb_root}/Tables/dbo/{table}/_delta_log")
    except Exception:
        return False

for t in shortcut_tables:
    r = client.get(f"/v1/workspaces/{ws_id}/items/{bronze_id}/shortcuts/Tables/{t}")
    if r.status_code == 200:
        print(f"[step] shortcut Tables/{t} already exists")
        continue

    deadline = time.time() + shortcut_wait_minutes * 60
    while not target_ready(t) and time.time() < deadline:
        print(f"[step] waiting for {sqldb_name} to replicate dbo.{t} to OneLake ...")
        time.sleep(30)

    body = {
        "path": "Tables",
        "name": t,
        "target": {"oneLake": {"workspaceId": ws_id, "itemId": sqldb_id, "path": f"Tables/dbo/{t}"}},
    }
    # VERIFY: shortcut body shape and the Tables/dbo/<table> target path for SQL database in Fabric.
    r = client.post(f"/v1/workspaces/{ws_id}/items/{bronze_id}/shortcuts", json=body)
    if r.status_code in (200, 201):
        print(f"[step] created shortcut Tables/{t} -> {sqldb_name}/Tables/dbo/{t}")
    elif r.status_code == 409:
        print(f"[step] shortcut Tables/{t} already exists (409)")
    else:
        raise RuntimeError(f"Shortcut {t} failed: HTTP {r.status_code} {r.text[:500]} - use the UI fallback above.")

In [ ]:
# SYNTHETIC DATA · for training only · no real customer data
# Quick check: read through the shortcuts
for t in shortcut_tables:
    n = spark.read.format("delta").load(f"{bronze_root}/Tables/{t}").count()
    print(f"[step] lh_bronze.{t} (shortcut): {n} rows")